# 12F-07 — Vehicle Demand engineering model

**Question:** How do roadload coefficients become steady-speed wheel demand?

In [ ]:
from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in search_roots if (path / "AGENTS.md").exists() and (path / "etl").exists())
import sqlite3
import numpy as np

CANONICAL_DB = REPO_ROOT / "etl/data/staging/sprint_12e2_epa_fuelcons/eco_drive_canonical_epa_fuelcons.db"
uri = CANONICAL_DB.resolve().as_uri() + "?mode=ro"
connection = sqlite3.connect(uri, uri=True)


## Load canonical VDE coefficients read-only

In [ ]:
query = '''
SELECT id AS vde_id, year, category, baseline_mass_kg,
       coast_A_N, coast_B_N_per_kph, coast_C_N_per_kph2,
       vde_total_mj_per_km, provenance_json
FROM vde
WHERE coast_A_N IS NOT NULL AND coast_B_N_per_kph IS NOT NULL AND coast_C_N_per_kph2 IS NOT NULL
ORDER BY id
LIMIT 100
'''
vdes = pd.read_sql_query(query, connection)
vdes.head()


## Make the roadload equation visible

At steady speed, `F = A + Bv + Cv²`. Dividing force by 3.6 converts N to Wh/km over one kilometre.

In [ ]:
assert not vdes.empty, "No VDE rows available for the notebook example."
selected = vdes.iloc[0]
speed_kph = np.arange(0, 131, 10)
force_n = selected["coast_A_N"] + selected["coast_B_N_per_kph"] * speed_kph + selected["coast_C_N_per_kph2"] * speed_kph**2

curve = pd.DataFrame({"speed_kph": speed_kph, "roadload_force_n": force_n})
curve["steady_speed_wh_per_km"] = curve["roadload_force_n"] / 3.6
curve


## A small coefficient sensitivity

In [ ]:
curve["plus_10pct_A_wh_per_km"] = (1.10 * selected["coast_A_N"] + selected["coast_B_N_per_kph"] * speed_kph + selected["coast_C_N_per_kph2"] * speed_kph**2) / 3.6
curve["plus_10pct_C_wh_per_km"] = (selected["coast_A_N"] + selected["coast_B_N_per_kph"] * speed_kph + 1.10 * selected["coast_C_N_per_kph2"] * speed_kph**2) / 3.6
curve.tail()


In [ ]:
import matplotlib.pyplot as plt

ax = curve.plot(x="speed_kph", y=["steady_speed_wh_per_km", "plus_10pct_A_wh_per_km", "plus_10pct_C_wh_per_km"], figsize=(7, 4))
ax.set(ylabel="Wh/km", title=f"Illustrative steady-speed demand: VDE {selected['vde_id']}")
plt.show()
connection.close()


## What did we learn?

A matters at every speed; C grows quadratically. This steady-speed slice is explanatory and is not a replacement for the production drive-cycle integration or TOTAL/NET contract.